# 00 – Pregătirea datelor (o singură dată)
Exact pașii 1–5 din `antrenare_yolo_segmentare_din_txt`: descărcare → duplicate → date noi → **dreptunghiuri → poligoane (GrabCut)** → verificare.
Rezultatul ajunge în `date/dataset_seg/` și îl folosesc toate cele 20 de experimente, deci toate se antrenează pe **exact aceleași date**.

- Dacă în folder există deja arhiva `dataset_segmentare_grabcut_cu_date_noi.zip` (cea din Google Drive → licenta), doar o dezarhivează (nu trebuie internet).
- Altfel descarcă din Roboflow (trebuie internet) și calculează poligoanele (~10–30 min).

Nu trebuie să-l pornești de mână: `python ruleaza_toate.py` îl rulează singur dacă datele lipsesc.

## 1. Setări

In [ ]:
import os, glob, shutil, random, json, time, yaml
import numpy as np
from pathlib import Path

# Aceleași setări ca în antrenare_yolo_segmentare_din_txt
ADAUGA_DATE_NOI = True       # True = și pozele din „dusty and bird-droppings”
METODA = "grabcut"           # "grabcut", "culoare" sau "dreptunghi"

BAZA = Path(os.getcwd()).resolve()          # folderul experimente_parametri
DATE = BAZA / "date"
LOGURI = BAZA / "loguri"
DATE.mkdir(exist_ok=True); LOGURI.mkdir(exist_ok=True)

SEG = str(DATE / "dataset_seg")
SUFIX = "_cu_date_noi" if ADAUGA_DATE_NOI else ""
NUME_ZIP = f"dataset_segmentare_{METODA}{SUFIX}.zip"
# Arhiva poate sta lângă notebook sau în folderul date/
ZIP_SEG = next((str(p) for p in (BAZA / NUME_ZIP, DATE / NUME_ZIP) if p.exists()), str(BAZA / NUME_ZIP))

# În repo arhiva e împărțită în bucăți (GitHub nu primește fișiere > 100 MB) → le lipim la loc
bucati = sorted(glob.glob(str(BAZA / f"{NUME_ZIP}.part*")))
if not os.path.exists(ZIP_SEG) and bucati:
    ZIP_SEG = str(DATE / NUME_ZIP)
    print(f"Lipesc {len(bucati)} bucăți ale arhivei → {ZIP_SEG}")
    with open(ZIP_SEG, "wb") as iesire:
        for b in bucati:
            with open(b, "rb") as intrare:
                shutil.copyfileobj(intrare, iesire, 16 * 1024 * 1024)
t_start = time.time()
print("Metodă:", METODA, "| date noi:", ADAUGA_DATE_NOI)
print("Arhivă:", ZIP_SEG, "→", "EXISTĂ (nu mai descarc nimic)" if os.path.exists(ZIP_SEG) else "nu există → descarc din Roboflow și calculez poligoanele")

## 2. Descărcare (doar dacă nu există arhiva)
Cheia Roboflow se citește din `cheie_roboflow.txt` sau din variabila `ROBOFLOW_API_KEY` – nu e scrisă în notebook.

In [ ]:
if os.path.exists(ZIP_SEG):
    print("✅ Arhiva există – sar peste descărcare (o dezarhivez la pasul 4).")
else:
    try:
        from roboflow import Roboflow
    except ImportError:
        raise SystemExit(f"Nu am nici arhiva {NUME_ZIP}, nici pachetul roboflow. "
                         "Pune arhiva (din Google Drive → licenta) în folderul experimente_parametri.")
    cale_cheie = BAZA / "cheie_roboflow.txt"
    CHEIE = os.environ.get("ROBOFLOW_API_KEY") or (cale_cheie.read_text().strip() if cale_cheie.exists() else "")
    if not CHEIE:
        raise SystemExit("Lipsește cheia Roboflow: pune-o în cheie_roboflow.txt (nu intră în git) "
                         "sau în variabila ROBOFLOW_API_KEY.")
    rf = Roboflow(api_key=CHEIE)
    project = rf.workspace("pintea-david").project("6rainstorm-final-project-ivbpe")
    dataset = project.version(1).download("yolov11", location=str(DATE / "original"))
    D = dataset.location

## 3. Scoatem duplicatele dintre seturi

In [ ]:
import glob, shutil
if not os.path.exists(ZIP_SEG):
    def baza(cale):
        return os.path.basename(cale).split(".rf.")[0]

    def muta(din, spre, baze_tinta):
        n = 0
        for img in glob.glob(f"{D}/{din}/images/*"):
            if baza(img) in baze_tinta:
                eticheta = img.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
                shutil.move(img, f"{D}/{spre}/images/")
                if os.path.exists(eticheta):
                    shutil.move(eticheta, f"{D}/{spre}/labels/")
                n += 1
        return n

    baze_train = {baza(p) for p in glob.glob(f"{D}/train/images/*")}
    print("test  → train:", muta("test", "train", baze_train))
    print("valid → train:", muta("valid", "train", baze_train))
    baze_valid = {baza(p) for p in glob.glob(f"{D}/valid/images/*")}
    print("test  → valid:", muta("test", "valid", baze_valid))
    for s in ["train", "valid", "test"]:
        print(f"{s:6s}: {len(os.listdir(f'{D}/{s}/images'))} imagini")

## 3b. Date noi: „dusty and bird-droppings”

In [ ]:
if ADAUGA_DATE_NOI and not os.path.exists(ZIP_SEG):
    import imagehash
    from PIL import Image as PILImage

    # --- descărcare (ultima versiune)
    proiect2 = rf.workspace("solar-panel-a5cyn").project("dusty-and-bird-droppings")
    versiuni = []
    try:
        versiuni = sorted(int(str(v.version).split("/")[-1]) for v in proiect2.versions())
    except Exception as e:
        print("Nu pot lista versiunile, le încerc pe rând:", type(e).__name__)
    ds2 = None
    for n in list(reversed(versiuni)) + list(range(10, 0, -1)):
        try:
            ds2 = proiect2.version(n).download("yolov11", location=str(DATE / "dataset_nou"))
            print("✅ Descărcată versiunea", n)
            break
        except Exception:
            continue
    D2 = ds2.location

    # --- redenumirea claselor (păstrăm toate coordonatele: dreptunghi SAU poligon)
    nume = yaml.safe_load(open(f"{D}/data.yaml"))["names"]
    nume_noi = yaml.safe_load(open(f"{D2}/data.yaml"))["names"]
    if isinstance(nume_noi, dict):
        nume_noi = [nume_noi[k] for k in sorted(nume_noi)]

    def clasa_mea(n):
        t = n.lower().replace("-", " ").replace("_", " ")
        if "bird" in t: return "Bird Drop"
        if "crack" in t and "dust" not in t: return "Physical Damage"
        if "dust" in t: return "Dust"
        if "snow" in t: return "Snow"
        return None

    mapare = {i: (nume.index(clasa_mea(n)) if clasa_mea(n) else None) for i, n in enumerate(nume_noi)}
    for i, n in enumerate(nume_noi):
        print(f"  {n!r:25} → {clasa_mea(n) or 'ELIMINAT'}")
    for f in glob.glob(f"{D2}/*/labels/*.txt"):
        linii = [" ".join([str(mapare[int(p[0])])] + p[1:])
                 for p in (l.split() for l in open(f)) if p and mapare.get(int(p[0])) is not None]
        open(f, "w").write("\n".join(linii) + ("\n" if linii else ""))

    # --- duplicate (amprentă perceptuală)
    PRAG = 6
    def amprenta(cale):
        return int(str(imagehash.phash(PILImage.open(cale).convert("RGB"))), 16)
    def biti_diferiti(a, valori):
        x = np.bitwise_xor(np.asarray(valori, dtype=np.uint64), np.uint64(a))
        try:
            return np.bitwise_count(x)
        except AttributeError:
            return np.unpackbits(x.view(np.uint8).reshape(-1, 8), axis=1).sum(1)

    amprente_vechi = np.array([amprenta(p) for p in glob.glob(f"{D}/*/images/*")], dtype=np.uint64)
    pastrate, amprente_pastrate = [], []
    dup_cu_vechi = dup_intre_noi = fara_eticheta = 0
    for p in sorted(glob.glob(f"{D2}/*/images/*")):
        lbl = p.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
        if not os.path.exists(lbl) or not open(lbl).read().strip():
            fara_eticheta += 1; continue
        a = amprenta(p)
        if biti_diferiti(a, amprente_vechi).min() <= PRAG:
            dup_cu_vechi += 1; continue
        if amprente_pastrate and biti_diferiti(a, amprente_pastrate).min() <= PRAG:
            dup_intre_noi += 1; continue
        pastrate.append(p); amprente_pastrate.append(a)
    print(f"Poze noi păstrate: {len(pastrate)} | duplicat cu ale tale: {dup_cu_vechi} | "
          f"duplicat între ele: {dup_intre_noi} | fără etichete: {fara_eticheta}")

    # --- împărțire 70 / 15 / 15 (seed 42, ca în exp3)
    grupuri = {}
    for p in pastrate:
        grupuri.setdefault(baza(p), []).append(p)
    chei = sorted(grupuri)
    random.seed(42); random.shuffle(chei)
    n_test = n_val = round(0.15 * len(chei))
    atribuire = {k: ("test" if i < n_test else "valid" if i < n_test + n_val else "train") for i, k in enumerate(chei)}
    for k, lista in grupuri.items():
        s = atribuire[k]
        for p in lista:
            shutil.copy(p, f"{D}/{s}/images/nou_{os.path.basename(p)}")
            lbl = p.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
            shutil.copy(lbl, f"{D}/{s}/labels/nou_" + os.path.basename(lbl))
    for s in ["train", "valid", "test"]:
        noi = len(glob.glob(f"{D}/{s}/images/nou_*"))
        print(f"{s:6s}: {len(os.listdir(f'{D}/{s}/images'))} imagini (din care noi: {noi})")

## 4. Dreptunghiuri → poligoane (GrabCut); poligoanele existente se păstrează

In [ ]:
import cv2, numpy as np, yaml
from concurrent.futures import ProcessPoolExecutor
from tqdm.auto import tqdm

MARCAJ = f"{SEG}/_procesate.txt"
CLASE_DESCHISE = {0, 5}          # Bird Drop, Snow – albe pe panou închis (folosite de METODA = "culoare")


def contur_grabcut(img, x1, y1, x2, y2):
    h, w = img.shape[:2]
    bw, bh = x2 - x1, y2 - y1
    if bw < 6 or bh < 6:
        return None
    m = int(0.1 * max(bw, bh)) + 5                     # margine de fundal în jurul dreptunghiului
    X1, Y1 = max(0, int(x1) - m), max(0, int(y1) - m)
    X2, Y2 = min(w, int(x2) + m), min(h, int(y2) + m)
    decupaj = img[Y1:Y2, X1:X2]
    rx, ry = int(x1) - X1, int(y1) - Y1
    rect = (max(rx, 1), max(ry, 1), max(min(int(bw), decupaj.shape[1] - max(rx, 1) - 1), 1),
            max(min(int(bh), decupaj.shape[0] - max(ry, 1) - 1), 1))
    masca = np.zeros(decupaj.shape[:2], np.uint8)
    try:
        cv2.grabCut(decupaj, masca, rect, np.zeros((1, 65)), np.zeros((1, 65)), 4, cv2.GC_INIT_WITH_RECT)
    except cv2.error:
        return None
    obiect = ((masca == cv2.GC_FGD) | (masca == cv2.GC_PR_FGD)).astype(np.uint8)
    cnt, _ = cv2.findContours(obiect, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnt:
        return None
    return max(cnt, key=cv2.contourArea).reshape(-1, 2).astype(np.float32) + [X1, Y1]


def contur_culoare(img, x1, y1, x2, y2):
    X1, Y1, X2, Y2 = int(max(x1, 0)), int(max(y1, 0)), int(x2), int(y2)
    decupaj = img[Y1:Y2, X1:X2]
    if decupaj.size == 0 or min(decupaj.shape[:2]) < 4:
        return None
    gri = cv2.GaussianBlur(cv2.cvtColor(decupaj, cv2.COLOR_BGR2GRAY), (3, 3), 0)
    _, alb = cv2.threshold(gri, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)   # prag automat (Otsu)
    alb = cv2.morphologyEx(alb, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    cnt, _ = cv2.findContours(alb, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnt:
        return None
    return max(cnt, key=cv2.contourArea).reshape(-1, 2).astype(np.float32) + [X1, Y1]


def poligon_din_contur(contur, w, h):
    if contur is None or len(contur) < 3:
        return None, 0.0
    simplu = cv2.approxPolyDP(np.asarray(contur, np.float32).reshape(-1, 1, 2), 1.0, True).reshape(-1, 2)
    if len(simplu) < 3:
        return None, 0.0
    norm = [f"{min(max(x / w, 0), 1):.5f} {min(max(y / h, 0), 1):.5f}" for x, y in simplu]
    return " ".join(norm), float(cv2.contourArea(simplu))


def proceseaza(f):
    """Primește un fișier .txt; întoarce (fișier, linii noi, statistici)."""
    stat = {"poligon_original": 0, "calculat": 0, "rezerva_dreptunghi": 0}
    linii = [l.split() for l in open(f).read().strip().splitlines() if l.strip()]
    stat["poligon_original"] = sum(len(p) > 5 for p in linii)
    dreptunghiuri = [(i, p) for i, p in enumerate(linii) if len(p) == 5]
    if not dreptunghiuri:
        return f, None, stat
    baza_img = f.replace("/labels/", "/images/").rsplit(".", 1)[0]
    cale = next((baza_img + e for e in (".jpg", ".jpeg", ".png") if os.path.exists(baza_img + e)), None)
    img = cv2.imread(cale) if cale else None
    if img is None:
        return f, None, stat
    h, w = img.shape[:2]
    for i, p in dreptunghiuri:
        cls = int(p[0])
        cx, cy, bw, bh = map(float, p[1:5])
        x1, y1, x2, y2 = (cx - bw/2) * w, (cy - bh/2) * h, (cx + bw/2) * w, (cy + bh/2) * h
        contur = None
        if METODA == "culoare" and cls in CLASE_DESCHISE:
            contur = contur_culoare(img, x1, y1, x2, y2)
        elif METODA in ("grabcut", "culoare"):
            contur = contur_grabcut(img, x1, y1, x2, y2)
        poligon, arie = poligon_din_contur(contur, w, h)
        arie_chenar = max((x2 - x1) * (y2 - y1), 1.0)
        if poligon and 0.05 * arie_chenar <= arie <= 0.97 * arie_chenar:
            linii[i] = [p[0]] + poligon.split()
            stat["calculat"] += 1
        else:   # rezervă: dreptunghiul ca poligon cu 4 colțuri
            linii[i] = [p[0]] + [f"{min(max(v, 0), 1):.5f}" for v in
                                 (x1/w, y1/h, x2/w, y1/h, x2/w, y2/h, x1/w, y2/h)]
            stat["rezerva_dreptunghi"] += 1
    return f, linii, stat


if os.path.exists(ZIP_SEG):
    if not os.path.exists(f"{SEG}/data.yaml"):
        shutil.unpack_archive(ZIP_SEG, SEG)
    print("✅ Dezarhivat:", SEG)
else:
    if not os.path.exists(MARCAJ):
        shutil.copytree(D, SEG, dirs_exist_ok=True)
        open(MARCAJ, "w").close()
    procesate = set(open(MARCAJ).read().split())
    fisiere = [f for f in sorted(glob.glob(f"{SEG}/*/labels/*.txt")) if f not in procesate]
    if procesate:
        print(f"Reiau: {len(procesate)} fișiere erau deja gata.")
    total = {"poligon_original": 0, "calculat": 0, "rezerva_dreptunghi": 0}
    import multiprocessing as mp
    paralel = "fork" in mp.get_all_start_methods()          # Linux: da; Windows: nu → rulează pe rând
    ex = ProcessPoolExecutor(max_workers=os.cpu_count(), mp_context=mp.get_context("fork")) if paralel else None
    with open(MARCAJ, "a") as marcaj:
        rezultate = ex.map(proceseaza, fisiere, chunksize=8) if ex else map(proceseaza, fisiere)
        bara = tqdm(rezultate, total=len(fisiere), desc=f"Poligoane ({METODA})", unit="fișier", mininterval=30)
        for f, linii, stat in bara:
            if linii is not None:
                open(f, "w").write("\n".join(" ".join(p) for p in linii) + "\n")
            marcaj.write(f + "\n")
            for k in total:
                total[k] += stat[k]
            bara.set_postfix(total)
    for c in glob.glob(f"{SEG}/*/labels.cache"):
        os.remove(c)
    print("Gata:", total)
    if ex: ex.shutdown()
    print("Arhivez (poate dura câteva minute, fără mesaje)…")
    os.remove(MARCAJ)
    shutil.make_archive(ZIP_SEG.replace(".zip", ""), "zip", SEG)
    print("✅ Arhivă salvată:", ZIP_SEG)

nume = yaml.safe_load(open(f"{SEG}/data.yaml"))["names"]
# Testul vechi (pozele tale) și testul nou (prefix nou_) – liste separate, pentru comparație
toate_test = sorted(os.path.basename(p) for p in glob.glob(f"{SEG}/test/images/*"))
open(f"{SEG}/test_vechi.txt", "w").write("\n".join(f"./test/images/{n}" for n in toate_test if not n.startswith("nou_")))
open(f"{SEG}/test_nou.txt", "w").write("\n".join(f"./test/images/{n}" for n in toate_test if n.startswith("nou_")))
baza_yaml = {"path": str(Path(SEG).resolve()), "train": "train/images", "val": "valid/images", "names": nume}
for fisier, test in [("data_seg.yaml", "test/images"), ("data_seg_test_vechi.yaml", "test_vechi.txt"),
                     ("data_seg_test_nou.yaml", "test_nou.txt")]:
    yaml.safe_dump({**baza_yaml, "test": test}, open(f"{SEG}/{fisier}", "w"), allow_unicode=True)
print(open(f"{SEG}/data_seg.yaml").read())
print("Test vechi:", sum(not n.startswith("nou_") for n in toate_test), "| test nou:", sum(n.startswith("nou_") for n in toate_test))

## 5. Verificare vizuală (se salvează și în `loguri/00_verificare_contururi.png`)

In [ ]:
import random, matplotlib.pyplot as plt
culori = [(255, 60, 60), (255, 140, 0), (60, 200, 230), (80, 200, 80), (60, 60, 255), (220, 220, 220)]

def deseneaza_contururi(cale_img, cale_lbl):
    img = cv2.cvtColor(cv2.imread(cale_img), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    strat = img.copy()
    for l in open(cale_lbl):
        p = l.split()
        if len(p) < 7:
            continue
        c = int(p[0])
        pts = (np.array(p[1:], dtype=float).reshape(-1, 2) * [w, h]).astype(np.int32)
        cv2.fillPoly(strat, [pts], culori[c % 6])
        cv2.polylines(img, [pts], True, culori[c % 6], 2)
        cv2.putText(img, nume[c], tuple(pts[0]), cv2.FONT_HERSHEY_SIMPLEX, 0.5, culori[c % 6], 2)
    return cv2.addWeighted(strat, 0.35, img, 0.65, 0)

random.seed(1)
poze = random.sample(glob.glob(f"{SEG}/train/images/*"), 9)
fig, axe = plt.subplots(3, 3, figsize=(15, 15))
for ax, cale in zip(axe.flat, poze):
    ax.imshow(deseneaza_contururi(cale, cale.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"))
    ax.set_title(os.path.basename(cale)[:30], fontsize=8); ax.axis("off")
plt.tight_layout(); plt.savefig(LOGURI / "00_verificare_contururi.png", dpi=80); plt.show()

noi = glob.glob(f"{SEG}/train/images/nou_*")
if noi:
    print("Doar poze NOI (din „dusty and bird-droppings”):")
    random.seed(2)
    fig, axe = plt.subplots(2, 3, figsize=(15, 10))
    for ax, cale in zip(axe.flat, random.sample(noi, min(6, len(noi)))):
        ax.imshow(deseneaza_contururi(cale, cale.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"))
        ax.axis("off")
    plt.tight_layout(); plt.savefig(LOGURI / "00_verificare_contururi_noi.png", dpi=80); plt.show()

## 6. Gata

In [ ]:
statistici = {s: len(glob.glob(f"{SEG}/{s}/images/*")) for s in ["train", "valid", "test"]}
statistici["test_vechi"] = sum(1 for l in open(f"{SEG}/test_vechi.txt") if l.strip())
statistici["test_nou"] = sum(1 for l in open(f"{SEG}/test_nou.txt") if l.strip())
rezultat = {"experiment": "00_pregatire_date", "stare": "gata", "metoda": METODA, "date_noi": ADAUGA_DATE_NOI,
            "poze": statistici, "durata_min": round((time.time() - t_start) / 60, 1)}
json.dump(rezultat, open(LOGURI / "00_pregatire_date.json", "w"), indent=2, ensure_ascii=False)
print("✅ Date pregătite:", statistici)
print("Uită-te la loguri/00_verificare_contururi.png ca să vezi dacă poligoanele cad pe defecte.")